# Sparse GP Regression with `gpc`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/sparse_gp_regression.ipynb)

GPc port of [`GPy/sparse_gp_regression.ipynb`](../GPy/sparse_gp_regression.ipynb) (James Hensman / Neil Lawrence).

We show the variational compression effect of sparse GPs under Titsias (2009). In GPc the default sparse approximation is **DTCVAR** (`approx="dtcvar"`), matching GPy's VarDTC spirit for this tutorial.

In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d

rng = np.random.default_rng(101)

## Sample function

Sample from an RBF prior on a 1-D grid (lengthscale/variance 1), with observation noise.

In [ ]:
N = 50
noise_var = 0.05
X = np.linspace(0, 10, N)[:, None]

def rbf_gram(X, lengthscale=1.0, variance=1.0):
    d2 = (X - X.T) ** 2
    return variance * np.exp(-0.5 * d2 / lengthscale**2)

K = rbf_gram(X)
y = rng.multivariate_normal(np.zeros(N), K + np.eye(N) * noise_var).reshape(-1, 1)

## Full Gaussian process fit

In [ ]:
m_full = gpc.models.GPRegression(X, y, kernel=gpc.kern.RBF(1))
m_full.optimize(iters=200)
plot_1d(m_full, X, y, label="full GP")
plt.title("Full GP")
plt.show()
print(m_full)
print("full objective (−log lik):", m_full.objective())

## A poor sparse GP fit

Inducing inputs clumped in two regions; hold them fixed and set likelihood noise near the truth.

In [ ]:
Z = np.hstack((np.linspace(2.5, 4.0, 3), np.linspace(7.0, 8.5, 3)))[:, None]
m = gpc.models.SparseGPRegression(X, y, Z=Z, kernel=gpc.kern.RBF(1), inducing_fixed=True)
m.set_by_path("likelihood.variance", noise_var)
plot_1d(m, X, y, label="sparse (fixed Z)")
plt.title("Sparse GP, fixed inducing (before optimize)")
plt.show()
print(m)

Fit is reasonable near inducing points and weak elsewhere.

### Optimizing covariance parameters (Z fixed)

In [ ]:
m.optimize(iters=200)
plot_1d(m, X, y, label="sparse (Z fixed)")
plt.title("After optimizing hypers with Z fixed")
plt.show()
print(m)
print("sparse objective:", m.objective())

Poor Z often lengthens the lengthscale / raises noise relative to the full GP.

### Optimizing inducing inputs

Rebuild with free inducing locations (`inducing_fixed=False`, the façade default):

In [ ]:
m = gpc.models.SparseGPRegression(X, y, Z=Z, kernel=gpc.kern.RBF(1), inducing_fixed=False)
m.optimize(iters=300)
plot_1d(m, X, y, label="sparse (free Z)")
plt.title("After optimizing hypers and Z")
plt.show()
print(m)
print("Z:\n", m.inducing_inputs())

### More inducing points

Twelve inducing inputs should tighten the approximation toward the full GP.

In [ ]:
Z12 = rng.random((12, 1)) * 12.0
m12 = gpc.models.SparseGPRegression(X, y, Z=Z12, kernel=gpc.kern.RBF(1))
m12.optimize(iters=300)
plot_1d(m12, X, y, label="sparse M=12")
plt.title("Sparse GP, M=12")
plt.show()
plot_1d(m_full, X, y, label="full GP")
plt.title("Full GP (reference)")
plt.show()
print("sparse M=12 objective:", m12.objective())
print("full objective:       ", m_full.objective())
# GPy compared log_likelihood (maximise); gpc.objective is −log lik (minimise).
print("gap (sparse − full) on −log lik:", m12.objective() - m_full.objective())